**Data Ingestion from Azure Blob Storage into Databricks**

Loading All Environmental Variables

In [0]:
import os
from dotenv import load_dotenv
load_dotenv('/Workspace/Repos/jagdisheverest23@gmail.com/DE-Mini-Project/.env', override=True)

storage_account_name = os.getenv('STORAGE_ACCOUNT_NAME')
storage_account_key = os.getenv('STORAGE_ACCOUNT_ACCESS_KEY')
storage_container_name = os.getenv('STORAGE_CONTAINER_NAME')
storage_account_connection_string = os.getenv('STORAGE_ACCOUNT_CONNECTION_STRING')

Collecting All Files in Azure Storage Account

In [0]:

from azure.storage.blob import BlobServiceClient

print(f"Files in container: {storage_container_name}")
print(f"Storage account: {storage_account_name}\n")

blob_service_client = BlobServiceClient.from_connection_string(storage_account_connection_string)
container_client = blob_service_client.get_container_client(storage_container_name)

csv_files = list(container_client.list_blobs())

print(f"Total files: {len(csv_files)}\n")

Loading All CSV Files into Bronze Catalog

In [0]:

for csv_file in csv_files:
    print(f"File: {csv_file.name}")

    base_path = f"abfss://{storage_container_name}@{storage_account_name}.dfs.core.windows.net/"
    file_path = f"{csv_file.name}"
    full_path = base_path + file_path

    df = (spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
        .load(full_path)
    )
    
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, col_name.replace(" ", "_"))
    
    source_path = f'/Volumes/bronze_raw_medical_data/azure_blob_storage/raw_medical_data/raw_{csv_file.name}'
    df.write.mode('overwrite').option('header',True).csv(source_path)
    print(f"Table: {source_path}")